In [1]:
import pandas as pd
import numpy as np

RAW = "../data/raw/"
PROCESSED = "../data/processed/"


customers   = pd.read_csv(RAW + "olist_customers_dataset.csv")
geolocation = pd.read_csv(RAW + "olist_geolocation_dataset.csv")
orders      = pd.read_csv(RAW + "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW + "olist_order_items_dataset.csv")
payments    = pd.read_csv(RAW + "olist_order_payments_dataset.csv")
reviews     = pd.read_csv(RAW + "olist_order_reviews_dataset.csv")
products    = pd.read_csv(RAW + "olist_products_dataset.csv")
sellers     = pd.read_csv(RAW + "olist_sellers_dataset.csv")
cat_trans   = pd.read_csv(RAW + "product_category_name_translation.csv")



In [2]:
geolocation = (
    geolocation
    .drop_duplicates(subset="geolocation_zip_code_prefix")
    .reset_index(drop=True)
)

In [3]:
date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")


In [4]:
print(orders["order_status"].value_counts())



order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [5]:
# ------------------------------------------------------------------
# 4. PRODUCTS — 610 lignes sans catégorie → on les catégorise "unknown"
#    plutôt que de les supprimer (on perdrait les ventes associées)
# ------------------------------------------------------------------
products["product_category_name"] = products["product_category_name"].fillna("unknown")
products[["product_name_lenght", "product_description_lenght", "product_photos_qty"]] = (
    products[["product_name_lenght", "product_description_lenght", "product_photos_qty"]].fillna(0)
)
# les 2 lignes sans poids/dimensions → medianes
for col in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    products[col] = products[col].fillna(products[col].median())


In [6]:

# ------------------------------------------------------------------
# 5. REVIEWS — on ne touche pas aux NaN de comment_title/message
#    (normal qu'un client ne laisse pas de commentaire),
#    on garde juste review_score qui est complet
# ------------------------------------------------------------------
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])



In [7]:
# ------------------------------------------------------------------
# 6. PAYMENTS / ORDER_ITEMS — 0 doublons, 0 NaN → rien à faire
#    juste vérifier les valeurs aberrantes (prix/freight négatifs ou à 0)
# ------------------------------------------------------------------
print("Prix <= 0 :", (order_items["price"] <= 0).sum())
print("Freight < 0 :", (order_items["freight_value"] < 0).sum())
print("Payment value <= 0 :", (payments["payment_value"] <= 0).sum())



Prix <= 0 : 0
Freight < 0 : 0
Payment value <= 0 : 9


In [9]:
# ------------------------------------------------------------------
# 7. EXPORT vers data/processed
# ------------------------------------------------------------------
customers.to_csv(PROCESSED + "customers_clean.csv", index=False)
geolocation.to_csv(PROCESSED + "geolocation_clean.csv", index=False)
orders.to_csv(PROCESSED + "orders_clean.csv", index=False)
order_items.to_csv(PROCESSED + "order_items_clean.csv", index=False)
payments.to_csv(PROCESSED + "payments_clean.csv", index=False)
reviews.to_csv(PROCESSED + "reviews_clean.csv", index=False)
products.to_csv(PROCESSED + "products_clean.csv", index=False)
sellers.to_csv(PROCESSED + "sellers_clean.csv", index=False)

print("✅ Nettoyage terminé, fichiers exportés dans data/processed/")

✅ Nettoyage terminé, fichiers exportés dans data/processed/
